# TradeEnv Hackathon Notebook

This notebook builds a minimal, professional baseline for bulk stock execution:
- Download 5 months of NIFTY data from yfinance
- Create a test-day execution simulator with max `k=10` trades/day
- Train a tiny deep-learning policy (2-layer MLP in NumPy)
- Compare policy vs random baseline on test days

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

SEED = 42
rng = np.random.default_rng(SEED)

NIFTY50 = [
    'RELIANCE.NS','TCS.NS','INFY.NS','HDFCBANK.NS','ICICIBANK.NS','SBIN.NS','ITC.NS',
    'LT.NS','KOTAKBANK.NS','HINDUNILVR.NS','AXISBANK.NS','ASIANPAINT.NS','BAJFINANCE.NS',
    'MARUTI.NS','SUNPHARMA.NS','TITAN.NS','ULTRACEMCO.NS','WIPRO.NS','NTPC.NS','POWERGRID.NS'
]

raw = yf.download(
    tickers=NIFTY50,
    period='5mo',
    interval='1d',
    auto_adjust=True,
    progress=False,
    group_by='ticker',
    threads=False,
)

frames = []
for sym in NIFTY50:
    if isinstance(raw.columns, pd.MultiIndex):
        if sym not in raw.columns.get_level_values(0):
            continue
        s = raw[sym].copy()
    else:
        s = raw.copy()
    if s.empty:
        continue
    s = s.rename(columns={c: c.lower() for c in s.columns})
    req = ['open', 'high', 'low', 'close', 'volume']
    if not set(req).issubset(s.columns):
        continue
    s = s[req].dropna().copy()
    s['symbol'] = sym
    frames.append(s)

df = pd.concat(frames).reset_index().rename(columns={'Date': 'date', 'index': 'date'})
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

def add_indicators(g):
    g = g.copy()
    g['ema_fast'] = g['close'].ewm(span=5, adjust=False).mean()
    g['ema_slow'] = g['close'].ewm(span=14, adjust=False).mean()
    d = g['close'].diff().fillna(0.0)
    up = d.clip(lower=0.0).rolling(14, min_periods=1).mean()
    dn = (-d.clip(upper=0.0)).rolling(14, min_periods=1).mean()
    rs = up / np.maximum(dn, 1e-9)
    g['rsi_14'] = 100 - (100 / (1 + rs))
    g['vwap_day_proxy'] = (g['close'] * g['volume']).cumsum() / np.maximum(g['volume'].cumsum(), 1.0)
    g['idx'] = np.arange(len(g))
    return g

df = df.groupby('symbol', group_keys=False).apply(add_indicators).reset_index(drop=True)
df = df[df['idx'] >= 14].reset_index(drop=True)
print('rows:', len(df), 'symbols:', df['symbol'].nunique())
df.head()

rows: 1760 symbols: 20


/var/folders/jb/2yhtg48x5zv4vdxrfxd3rjg80000gp/T/ipykernel_42410/950157423.py:59: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('symbol', group_keys=False).apply(add_indicators).reset_index(drop=True)


Price,date,open,high,low,close,volume,symbol,ema_fast,ema_slow,rsi_14,vwap_day_proxy,idx
0,2025-11-28,2880.000000,2889.899902,2855.800049,2874.399902,748446,ASIANPAINT.NS,2875.094779,2840.241751,80.234102,2850.700727,14
1,2025-12-01,2877.699951,2887.500000,2848.000000,2867.600098,499153,ASIANPAINT.NS,2872.596552,2843.889530,78.432343,2850.986851,15
2,2025-12-02,2867.000000,2962.000000,2861.199951,2954.399902,2102147,ASIANPAINT.NS,2899.864335,2858.624246,76.860727,2857.869833,16
3,2025-12-03,2969.000000,2969.199951,2932.600098,2953.500000,1252456,ASIANPAINT.NS,2917.742890,2871.274347,66.161574,2861.517425,17
4,2025-12-04,2950.000000,2985.699951,2934.000000,2957.199951,1466098,ASIANPAINT.NS,2930.895244,2882.731094,62.582127,2865.606961,18


In [2]:
def synthetic_path(row, n=10):
    x = np.array([0.0, 0.35, 0.7, 1.0])
    y = np.array([row.open, row.low, row.high, row.close], dtype=float)
    return np.interp(np.linspace(0, 1, n), x, y)

def build_dataset(frame):
    feats, labels = [], []
    for r in frame.itertuples(index=False):
        path = synthetic_path(r, n=10)
        spread = max(r.high - r.low, 1e-6)
        for t, px in enumerate(path):
            progress = t / 9
            f = [
                (px - r.vwap_day_proxy) / max(r.vwap_day_proxy, 1e-6),
                (r.ema_fast - r.ema_slow) / max(r.ema_slow, 1e-6),
                (r.rsi_14 - 50.0) / 50.0,
                (px - r.low) / spread,
                progress,
            ]
            # teacher signal: larger buy when relative price is cheap
            y = 1.0 / (1.0 + np.exp(8.0 * f[0]))
            feats.append(f)
            labels.append([y])
    return np.array(feats, dtype=np.float32), np.array(labels, dtype=np.float32)

split_date = df['date'].quantile(0.75)
train_df = df[df['date'] <= split_date].copy()
test_df = df[df['date'] > split_date].copy()

X_train, y_train = build_dataset(train_df)
X_test, y_test = build_dataset(test_df)
X_train.shape, X_test.shape

((13200, 5), (4400, 5))

In [3]:
# Tiny DL policy: 2-layer MLP in NumPy (minimal, no heavy dependencies)
d_in, d_h = X_train.shape[1], 16
W1 = rng.normal(0, 0.2, size=(d_in, d_h)).astype(np.float32)
b1 = np.zeros((1, d_h), dtype=np.float32)
W2 = rng.normal(0, 0.2, size=(d_h, 1)).astype(np.float32)
b2 = np.zeros((1, 1), dtype=np.float32)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

lr = 0.03
for epoch in range(200):
    z1 = X_train @ W1 + b1
    h1 = np.tanh(z1)
    z2 = h1 @ W2 + b2
    yhat = sigmoid(z2)

    loss = np.mean((yhat - y_train) ** 2)

    # backprop (MSE + sigmoid output)
    dz2 = (2.0 / len(X_train)) * (yhat - y_train) * yhat * (1 - yhat)
    dW2 = h1.T @ dz2
    db2 = dz2.sum(axis=0, keepdims=True)

    dh1 = dz2 @ W2.T
    dz1 = dh1 * (1 - np.tanh(z1) ** 2)
    dW1 = X_train.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)

    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

    if epoch % 50 == 0:
        print(f'epoch={epoch:03d} loss={loss:.6f}')

def policy_fraction(x):
    h = np.tanh(x @ W1 + b1)
    return float(sigmoid(h @ W2 + b2).squeeze())

epoch=000 loss=0.008912
epoch=050 loss=0.008731
epoch=100 loss=0.008556
epoch=150 loss=0.008386


In [4]:
def run_day(row, mode='dl', target_qty=1000, k=10):
    path = synthetic_path(row, n=10)
    remaining = target_qty
    trades = 0
    notional = 0.0

    for t, px in enumerate(path):
        if remaining <= 0:
            break
        f = np.array([[
            (px - row.vwap_day_proxy) / max(row.vwap_day_proxy, 1e-6),
            (row.ema_fast - row.ema_slow) / max(row.ema_slow, 1e-6),
            (row.rsi_14 - 50.0) / 50.0,
            (px - row.low) / max(row.high - row.low, 1e-6),
            t / 9.0,
        ]], dtype=np.float32)

        if mode == 'dl':
            frac = policy_fraction(f)
        else:
            frac = rng.uniform(0, 1)

        if trades < k:
            q = int(min(remaining, max(0, frac * (target_qty / k))))
            if q > 0:
                trades += 1
                notional += q * px
                remaining -= q

    if remaining > 0:  # forced terminal fill
        notional += remaining * path[-1]
        remaining = 0

    avg_buy = notional / target_qty
    # score in [0,1]: lower avg buy is better
    score = (row.high - avg_buy) / max(row.high - row.low, 1e-6)
    score = float(np.clip(score, 0.0, 1.0))
    return avg_buy, score, trades

rows = []
for r in test_df.itertuples(index=False):
    dl_buy, dl_score, dl_trades = run_day(r, mode='dl', k=10)
    rd_buy, rd_score, rd_trades = run_day(r, mode='random', k=10)
    rows.append({
        'symbol': r.symbol, 'date': r.date,
        'dl_avg_buy': dl_buy, 'rand_avg_buy': rd_buy,
        'dl_score': dl_score, 'rand_score': rd_score,
        'dl_trades': dl_trades, 'rand_trades': rd_trades,
    })

res = pd.DataFrame(rows)
summary = pd.DataFrame({
    'metric': ['mean_score', 'median_score', 'mean_avg_buy', 'mean_trades'],
    'DL_policy': [res.dl_score.mean(), res.dl_score.median(), res.dl_avg_buy.mean(), res.dl_trades.mean()],
    'Random': [res.rand_score.mean(), res.rand_score.median(), res.rand_avg_buy.mean(), res.rand_trades.mean()],
})
summary

,metric,DL_policy,Random
0,mean_score,0.507174,0.509097
1,median_score,0.509686,0.515227
2,mean_avg_buy,2493.898263,2493.635019
3,mean_trades,10.000000,9.915909


In [5]:
print('DL beats random on score:', (res.dl_score.mean() > res.rand_score.mean()))
display(res.head(10))
display(summary)

DL beats random on score: False


,symbol,date,dl_avg_buy,rand_avg_buy,dl_score,rand_score,dl_trades,rand_trades
0,ASIANPAINT.NS,2026-03-05,2285.781444,2285.119172,0.458997,0.470165,10,10
1,ASIANPAINT.NS,2026-03-06,2277.672775,2278.951908,0.557697,0.534096,10,10
2,ASIANPAINT.NS,2026-03-09,2210.543333,2211.056165,0.288675,0.281066,10,10
3,ASIANPAINT.NS,2026-03-10,2276.325407,2276.620740,0.537845,0.532324,10,10
4,ASIANPAINT.NS,2026-03-11,2247.588349,2246.219259,0.715943,0.733957,10,10
5,ASIANPAINT.NS,2026-03-12,2220.756905,2219.719446,0.595973,0.617188,10,10
6,ASIANPAINT.NS,2026-03-13,2199.967017,2199.532015,0.664410,0.679056,10,10
7,ASIANPAINT.NS,2026-03-16,2208.867917,2208.524819,0.359244,0.365754,10,10
8,ASIANPAINT.NS,2026-03-17,2236.057392,2233.748913,0.574325,0.626435,10,10
9,ASIANPAINT.NS,2026-03-18,2255.094693,2254.996157,0.366155,0.368512,10,10


,metric,DL_policy,Random
0,mean_score,0.507174,0.509097
1,median_score,0.509686,0.515227
2,mean_avg_buy,2493.898263,2493.635019
3,mean_trades,10.000000,9.915909
